 ### Tareas de limpieza

 - Filtrar fechas raras (las de 2008)
 - Droppear filas innecesarias
 - Manejar nulos en las filas que se necesitan
 - Parseas fechas a datetime si es necesario

### análisis

In [16]:
import polars as pl

parquet_files = [
    'yellow_tripdata_2026-01.parquet',
    'yellow_tripdata_2026-02.parquet',
    'yellow_tripdata_2026-03.parquet'
]

for file_name in parquet_files:
    print(f"\n{'='*80}")
    print(f"ARCHIVO: {file_name}")
    print(f"{'='*80}")
    
    try:
        lazy_df = pl.scan_parquet(file_name)
        
        num_rows = lazy_df.select(pl.count()).collect().item()
        print(f"\n✓ Total de filas: {num_rows:,}")

        print(f"\n--- VALORES FALTANTES (NULL VALUES) ---")
        columns = lazy_df.collect_schema().names()
        null_counts = lazy_df.select([
            pl.col(col).is_null().sum().alias(col) for col in columns
        ]).collect().to_dict(as_series=False)
        
        null_data = []
        for col in columns:
            null_count = null_counts[col][0]
            null_pct = (null_count / num_rows) * 100 if num_rows > 0 else 0
            null_data.append({
                'Columna': col,
                'Nulos': null_count,
                'Porcentaje': f"{null_pct:.2f}%"
            })
        
        null_df = pl.DataFrame(null_data)
        print(null_df)
        
        print(f"\n--- RANGOS DE FECHAS (para verificar anomalías) ---")
        date_range = lazy_df.select([
            pl.col('tpep_pickup_datetime').min().alias('MIN_pickup'),
            pl.col('tpep_pickup_datetime').max().alias('MAX_pickup'),
            pl.col('tpep_dropoff_datetime').min().alias('MIN_dropoff'),
            pl.col('tpep_dropoff_datetime').max().alias('MAX_dropoff'),
        ]).collect()
        print(date_range)
        
        print(f"\n--- VERIFICACIÓN DE ANOMALÍAS ---")
        numeric_cols = ['trip_distance', 'fare_amount', 'passenger_count']
        anomalies = lazy_df.select([
            pl.col('trip_distance').min().alias('MIN_trip_distance'),
            pl.col('trip_distance').max().alias('MAX_trip_distance'),
            pl.col('fare_amount').min().alias('MIN_fare_amount'),
            pl.col('fare_amount').max().alias('MAX_fare_amount'),
            pl.col('passenger_count').min().alias('MIN_passengers'),
            pl.col('passenger_count').max().alias('MAX_passengers'),
        ]).collect()
        print(anomalies)
        
        
    except Exception as e:
        print(f"\nERROR al procesar {file_name}: {e}")


ARCHIVO: yellow_tripdata_2026-01.parquet

✓ Total de filas: 3,724,889

--- VALORES FALTANTES (NULL VALUES) ---
shape: (20, 3)
┌───────────────────────┬─────────┬────────────┐
│ Columna               ┆ Nulos   ┆ Porcentaje │
│ ---                   ┆ ---     ┆ ---        │
│ str                   ┆ i64     ┆ str        │
╞═══════════════════════╪═════════╪════════════╡
│ VendorID              ┆ 0       ┆ 0.00%      │
│ tpep_pickup_datetime  ┆ 0       ┆ 0.00%      │
│ tpep_dropoff_datetime ┆ 0       ┆ 0.00%      │
│ passenger_count       ┆ 1088058 ┆ 29.21%     │
│ trip_distance         ┆ 0       ┆ 0.00%      │
│ …                     ┆ …       ┆ …          │
│ improvement_surcharge ┆ 0       ┆ 0.00%      │
│ total_amount          ┆ 0       ┆ 0.00%      │
│ congestion_surcharge  ┆ 1088058 ┆ 29.21%     │
│ Airport_fee           ┆ 1088058 ┆ 29.21%     │
│ cbd_congestion_fee    ┆ 0       ┆ 0.00%      │
└───────────────────────┴─────────┴────────────┘

--- RANGOS DE FECHAS (para verificar an

C:\Users\PC\AppData\Local\Temp\ipykernel_7636\814742264.py:17: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  num_rows = lazy_df.select(pl.count()).collect().item()


shape: (20, 3)
┌───────────────────────┬─────────┬────────────┐
│ Columna               ┆ Nulos   ┆ Porcentaje │
│ ---                   ┆ ---     ┆ ---        │
│ str                   ┆ i64     ┆ str        │
╞═══════════════════════╪═════════╪════════════╡
│ VendorID              ┆ 0       ┆ 0.00%      │
│ tpep_pickup_datetime  ┆ 0       ┆ 0.00%      │
│ tpep_dropoff_datetime ┆ 0       ┆ 0.00%      │
│ passenger_count       ┆ 1023317 ┆ 30.10%     │
│ trip_distance         ┆ 0       ┆ 0.00%      │
│ …                     ┆ …       ┆ …          │
│ improvement_surcharge ┆ 0       ┆ 0.00%      │
│ total_amount          ┆ 0       ┆ 0.00%      │
│ congestion_surcharge  ┆ 1023317 ┆ 30.10%     │
│ Airport_fee           ┆ 1023317 ┆ 30.10%     │
│ cbd_congestion_fee    ┆ 0       ┆ 0.00%      │
└───────────────────────┴─────────┴────────────┘

--- RANGOS DE FECHAS (para verificar anomalías) ---
shape: (1, 4)
┌─────────────────────┬─────────────────────┬─────────────────────┬──────────────────

In [10]:
lazy_df_03 = pl.scan_parquet('yellow_tripdata_2026-03.parquet')
date_check = lazy_df_03.select([
    pl.col('tpep_pickup_datetime').min().alias('min_date'),
    pl.col('tpep_pickup_datetime').max().alias('max_date'),
]).collect()

print(f"Mínima fecha en archivo 2026-03: {date_check['min_date'][0]}")
print(f"Máxima fecha en archivo 2026-03: {date_check['max_date'][0]}")

out_of_range = lazy_df_03.filter(
    (pl.col('tpep_pickup_datetime') < pl.datetime(2026, 1, 1)) |
    (pl.col('tpep_pickup_datetime') > pl.datetime(2026, 3, 31))
).select(pl.len()).collect().item()

total_rows = lazy_df_03.select(pl.len()).collect().item()

print(f"   - Total de filas en archivo 03: {total_rows:,}")
print(f"   - Filas FUERA del rango esperado (ene-mar 2026): {out_of_range:,}")
print(f"   - Porcentaje fuera de rango: {(out_of_range/total_rows)*100:.2f}%")

if out_of_range > 0:
    print(f"\nHAY {out_of_range:,} FILAS CON FECHAS INVÁLIDAS")
else:
    pass

files = [
    'yellow_tripdata_2026-01.parquet',
    'yellow_tripdata_2026-02.parquet',
    'yellow_tripdata_2026-03.parquet'
]

print(f"\n\n TABLA DE VALORES FALTANTES (TODOS LOS ARCHIVOS)")
print("-" * 80)

all_nulls = []
for file_name in files:
    lazy = pl.scan_parquet(file_name)
    cols = lazy.collect_schema().names()
    num_rows = lazy.select(pl.len()).collect().item()
    
    null_counts = lazy.select([
        pl.col(col).is_null().sum().alias(f'{col}_null_count') for col in cols
    ]).collect()

    for col in cols:
        null_count = null_counts[f'{col}_null_count'][0]
        if null_count > 0:
            null_pct = (null_count / num_rows) * 100
            all_nulls.append({
                'Archivo': file_name.replace('yellow_tripdata_', '').replace('.parquet', ''),
                'Columna': col,
                'Nulos': null_count,
                'Porcentaje': f"{null_pct:.1f}%"
            })

if all_nulls:
    null_df = pl.DataFrame(all_nulls)
    print(null_df)
else:
    pass

Mínima fecha en archivo 2026-03: 2008-12-31 23:03:20
Máxima fecha en archivo 2026-03: 2026-04-01 00:06:25
   - Total de filas en archivo 03: 3,952,451
   - Filas FUERA del rango esperado (ene-mar 2026): 114,558
   - Porcentaje fuera de rango: 2.90%

HAY 114,558 FILAS CON FECHAS INVÁLIDAS


 TABLA DE VALORES FALTANTES (TODOS LOS ARCHIVOS)
--------------------------------------------------------------------------------
shape: (15, 4)
┌─────────┬──────────────────────┬─────────┬────────────┐
│ Archivo ┆ Columna              ┆ Nulos   ┆ Porcentaje │
│ ---     ┆ ---                  ┆ ---     ┆ ---        │
│ str     ┆ str                  ┆ i64     ┆ str        │
╞═════════╪══════════════════════╪═════════╪════════════╡
│ 2026-01 ┆ passenger_count      ┆ 1088058 ┆ 29.2%      │
│ 2026-01 ┆ RatecodeID           ┆ 1088058 ┆ 29.2%      │
│ 2026-01 ┆ store_and_fwd_flag   ┆ 1088058 ┆ 29.2%      │
│ 2026-01 ┆ congestion_surcharge ┆ 1088058 ┆ 29.2%      │
│ 2026-01 ┆ Airport_fee          ┆ 1088058

In [11]:
files = ['yellow_tripdata_2026-01.parquet', 'yellow_tripdata_2026-02.parquet', 'yellow_tripdata_2026-03.parquet']

for f in files:
    try:
        df = pl.scan_parquet(f)
        rows = df.select(pl.len()).collect().item()
        date_range = df.select(
            pl.col('tpep_pickup_datetime').min().alias('min_d'),
            pl.col('tpep_pickup_datetime').max().alias('max_d')
        ).collect()
        
        print(f"\n  {f}")
        print(f"   Filas: {rows:,}")
        print(f"   Rango fechas: {date_range['min_d'][0]} a {date_range['max_d'][0]}")
    except Exception as e:
        print(f"  ERROR: {str(e)[:80]}")

print("-" * 100)

for f in files:
    df = pl.scan_parquet(f)
    cols = df.collect_schema().names()
    
    nulls_found = df.select([
        pl.col(c).is_null().sum().alias(c) for c in cols
    ]).collect().row(0)
    
    cols_with_nulls = [cols[i] for i, v in enumerate(nulls_found) if v > 0]
    
    if cols_with_nulls:
        print(f"\n{f}:")
        for col in cols_with_nulls:
            idx = cols.index(col)
            print(f"   • {col}: {nulls_found[idx]:,} nulos")
    else:
        print(f"\n{f}: Sin nulos")


  yellow_tripdata_2026-01.parquet
   Filas: 3,724,889
   Rango fechas: 2025-12-31 23:57:29 a 2026-02-01 00:45:01

  yellow_tripdata_2026-02.parquet
   Filas: 3,399,866
   Rango fechas: 2026-01-31 23:31:23 a 2026-03-01 00:51:48

  yellow_tripdata_2026-03.parquet
   Filas: 3,952,451
   Rango fechas: 2008-12-31 23:03:20 a 2026-04-01 00:06:25
----------------------------------------------------------------------------------------------------

yellow_tripdata_2026-01.parquet:
   • passenger_count: 1,088,058 nulos
   • RatecodeID: 1,088,058 nulos
   • store_and_fwd_flag: 1,088,058 nulos
   • congestion_surcharge: 1,088,058 nulos
   • Airport_fee: 1,088,058 nulos

yellow_tripdata_2026-02.parquet:
   • passenger_count: 1,023,317 nulos
   • RatecodeID: 1,023,317 nulos
   • store_and_fwd_flag: 1,023,317 nulos
   • congestion_surcharge: 1,023,317 nulos
   • Airport_fee: 1,023,317 nulos

yellow_tripdata_2026-03.parquet:
   • passenger_count: 945,748 nulos
   • RatecodeID: 945,748 nulos
   • store

### combinación de los parquets

In [15]:
files = [
    'yellow_tripdata_2026-01.parquet',
    'yellow_tripdata_2026-02.parquet',
    'yellow_tripdata_2026-03.parquet'
]

dfs = [pl.scan_parquet(f) for f in files]
combined = pl.concat(dfs)

print(f"Total de filas (antes de guardar): {combined.select(pl.len()).collect().item():,}")

output_file = 'yellow_tripdata_2026.parquet'

combined.collect().write_parquet(output_file)

check = pl.scan_parquet(output_file)
check_rows = check.select(pl.len()).collect().item()
print(f"\n {check_rows:,} filas en archivo combinado")


Total de filas (antes de guardar): 11,077,206

 11,077,206 filas en archivo combinado
